# Stage B continuation: router only

This notebook resumes a stopped 01b run from its saved output. It reuses the completed fixed adapter, route bank, alpha calibration, static-bank choice, norm files, and rollout shards. It runs **only router teacher scoring and router fitting**, then packages the preserved old files together with the new router artifacts as `ars_training_bundle.zip`.

**Important:** the current teacher-scoring loop writes its target checkpoint only after all selected questions have been scored. A run stopped at 3% cannot resume from that exact forward; this notebook starts the missing teacher-scoring stage again, with a smaller question cap. It does not retrain the bank or repeat static-bank/alpha selection.


## Input checklist

Attach the **saved output of the interrupted Notebook 01b run** as a Kaggle input (Notebook output or extracted Model input). Its root should contain `manifest.json` and these files:

- `checkpoints/fixed_route_adapter.pt`
- `checkpoints/session3_steering_bank.pt`
- `calibration/fixed_route_p99_norms.json` and `calibration/candidate_norms_p99.json`
- `training/rollouts/shard0.jsonl` and `training/rollouts/shard1.jsonl`
- `alpha_calibration.json` and `session3_static_bank_action.json`
- the `code/` snapshot recorded by the manifest

The notebook validates hashes before copying the input to a new writable run folder. The Kaggle input stays untouched. If Kaggle finds multiple matching run folders, set `STAGE_B_OUTPUT_OVERRIDE` to the exact saved 01b run folder.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Router teacher scoring uses one GPU.
WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
STAGE_B_OUTPUT_OVERRIDE = None  # Optional exact folder containing manifest.json and checkpoints/.
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _is_stage_a_manifest(value):
    return isinstance(value, dict) and value.get("format") == "ars_fixed_adapter_handoff_v1" and value.get("fixed_adapter_ready") is True

def _validate_stage_a_root(root):
    root = Path(root).resolve()
    manifest_path = root / "manifest.json"
    if not manifest_path.is_file():
        return False
    try:
        manifest = _read_json(manifest_path)
    except (OSError, ValueError, TypeError):
        return False
    if not _is_stage_a_manifest(manifest):
        return False
    required = [
        "checkpoints/fixed_route_adapter.pt", "checkpoints/session3_steering_bank.pt",
        "calibration/fixed_route_p99_norms.json", "calibration/candidate_norms_p99.json",
        "training/rollouts/shard0.jsonl", "training/rollouts/shard1.jsonl",
        "alpha_calibration.json", "session3_static_bank_action.json",
    ]
    return all((root / item).is_file() and (root / item).stat().st_size for item in required)

if STAGE_B_OUTPUT_OVERRIDE:
    SOURCE_RUN_DIR = Path(STAGE_B_OUTPUT_OVERRIDE).expanduser().resolve()
    if not _validate_stage_a_root(SOURCE_RUN_DIR):
        raise ValueError(f"The override is not a complete interrupted 01b output folder: {SOURCE_RUN_DIR}")
else:
    roots = set()
    for bank_path in INPUT_ROOT.rglob("session3_steering_bank.pt"):
        candidate = bank_path.parent.parent
        if _validate_stage_a_root(candidate):
            roots.add(candidate.resolve())
    if len(roots) != 1:
        raise FileNotFoundError(
            "Expected exactly one interrupted 01b output with its bank, fixed adapter, norms, "
            f"rollouts, alpha report, static action, and manifest; found: {sorted(map(str, roots))}. "
            "Attach that saved output or set STAGE_B_OUTPUT_OVERRIDE."
        )
    SOURCE_RUN_DIR = next(iter(roots))

SOURCE_MANIFEST = _read_json(SOURCE_RUN_DIR / "manifest.json")
if SOURCE_MANIFEST.get("architecture_id") != "ars.fixed_route_low_rank_residual" or SOURCE_MANIFEST.get("architecture_version") != "1.0.0":
    raise ValueError("The saved handoff uses an unsupported fixed-adapter architecture contract")

def _verify_source_manifest(root, manifest):
    expected = {}
    for section in ("artifacts", "rollouts", "collection_metadata_hashes", "code"):
        entries = manifest.get(section, {})
        if not isinstance(entries, dict):
            raise ValueError(f"Invalid {section} section in source manifest")
        expected.update(entries)
    if not expected:
        raise ValueError("Source manifest has no checksums to validate")
    errors = []
    for relative, metadata in expected.items():
        member = PurePosixPath(relative)
        checksum = metadata if isinstance(metadata, str) else metadata.get("sha256") if isinstance(metadata, dict) else None
        path = (root / Path(*member.parts)).resolve()
        if member.is_absolute() or ".." in member.parts or not isinstance(checksum, str):
            errors.append(f"invalid manifest entry: {relative}")
        elif not path.is_file() or sha256_file(path) != checksum.lower():
            errors.append(f"missing or checksum mismatch: {relative}")
    if errors:
        raise ValueError("Saved 01b input failed source-manifest verification: " + "; ".join(errors[:12]))

_verify_source_manifest(SOURCE_RUN_DIR, SOURCE_MANIFEST)
CONTINUATION_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
SOURCE_RUN_ID = SOURCE_RUN_DIR.name
RUN_ID = f"{SOURCE_RUN_ID}_router_resume_{CONTINUATION_ID}"
RUN_DIR = WORK_ROOT / "ars_session3_stage_b_resumed" / RUN_ID
if RUN_DIR.exists():
    raise FileExistsError(f"Refusing to overwrite an existing continuation output: {RUN_DIR}")
shutil.copytree(SOURCE_RUN_DIR, RUN_DIR)
print({"source_output": str(SOURCE_RUN_DIR), "continuation_output": str(RUN_DIR), "source_run": SOURCE_RUN_ID, "copied_old_output": True}, flush=True)


## Load the pinned training code

The saved handoff records the exact code commit and per-file checksums. This cell checks out that commit and verifies the code snapshot before running the router stage, so the continuation uses the same training implementation as the interrupted run.


In [ ]:
commit = SOURCE_MANIFEST["code_commit"]
REPO_DIR = WORK_ROOT / f"adaptive-residual-steering-router-{CONTINUATION_ID}"
subprocess.run(["git", "clone", "--branch", "development", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", commit], check=True)
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

code_errors = []
for relative, metadata in SOURCE_MANIFEST.get("code", {}).items():
    source_snapshot = RUN_DIR / relative
    repository_file = REPO_DIR / Path(relative).relative_to("code")
    expected = metadata.get("sha256") if isinstance(metadata, dict) else None
    if not expected or not source_snapshot.is_file() or sha256_file(source_snapshot) != expected:
        code_errors.append(f"saved snapshot mismatch: {relative}")
    if not repository_file.is_file() or sha256_file(repository_file) != expected:
        code_errors.append(f"pinned repository mismatch: {relative}")
if code_errors:
    raise ValueError("Pinned code validation failed: " + "; ".join(code_errors[:12]))

CHECKPOINT = RUN_DIR / "checkpoints/fixed_route_adapter.pt"
BANK_CHECKPOINT = RUN_DIR / "checkpoints/session3_steering_bank.pt"
NORM_FILE = RUN_DIR / "calibration/fixed_route_p99_norms.json"
CANDIDATE_NORMS = RUN_DIR / "calibration/candidate_norms_p99.json"
ROLLOUT_FILES = [RUN_DIR / "training/rollouts/shard0.jsonl", RUN_DIR / "training/rollouts/shard1.jsonl"]
ALPHA_REPORT = RUN_DIR / "alpha_calibration.json"
STATIC_ACTION = RUN_DIR / "session3_static_bank_action.json"
ROUTER_CHECKPOINT = RUN_DIR / "checkpoints/session3_router.pt"
ROUTER_REPORT = RUN_DIR / "checkpoints/session3_router_training.json"
TEACHER_TARGETS = RUN_DIR / "checkpoints/session3_route_teacher_targets.pt"
print({"pinned_commit": commit, "verified_code_files": len(SOURCE_MANIFEST["code"]), "router_code": str(REPO_DIR / "scripts/run_session3_router.py")}, flush=True)


## Validate reusable artifacts and set the runtime cap

The interrupted run evaluated 121,348 teacher forwards for about 1,345 available training questions. At the observed 4.9 forwards/second, a 400-question cap projects to about 2.0 hours of teacher scoring; router fitting, validation, and packaging add overhead. This targets roughly **2.5 hours total**, but GPU speed and the exact rollout groups can change the estimate.

`MAX_ROUTER_ROWS` is named “rows” in the CLI, but the trainer uses it as a cap on **unique training question IDs**. Bank training, alpha calibration, and static-bank screening are reused and are not repeated.


In [ ]:
import torch
import math
from src.steering.route_bank import normalize_route_catalog, validate_route_expert_states

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for router teacher scoring")
GPU_COUNT = torch.cuda.device_count()
CANDIDATE_LAYERS = list(SOURCE_MANIFEST["candidate_layers"])
FIXED_ROUTE = list(SOURCE_MANIFEST["route"])
COLLECTION_POLICY = SOURCE_MANIFEST["collection_policy"]
MAX_ROUTER_QUESTIONS = min(2000, int(COLLECTION_POLICY["questions"]))
MAX_ROUTER_ROWS = min(400, MAX_ROUTER_QUESTIONS)
ROUTER_EPOCHS = 3
ROUTER_ALPHAS = [1.0, 1.25, 1.5, 1.75, 2.0]
TASK_VALIDATION_ROWS = 84
ROUTE_COST = 0.01
SEED = int(SOURCE_MANIFEST.get("seed", 42))

for path in [CHECKPOINT, BANK_CHECKPOINT, NORM_FILE, CANDIDATE_NORMS, ALPHA_REPORT, STATIC_ACTION, *ROLLOUT_FILES]:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Required saved 01b artifact is missing or empty: {path}")

def _load_p99_norms(path, required_layers):
    payload = json.loads(path.read_text(encoding="utf-8"))
    layers = payload.get("layers", {})
    missing = []
    values = {}
    for layer in required_layers:
        stats = layers.get(str(layer), layers.get(layer))
        value = stats.get("p99_norm") if isinstance(stats, dict) else None
        if value is None or not math.isfinite(float(value)) or float(value) <= 0:
            missing.append(layer)
        else:
            values[int(layer)] = float(value)
    if missing:
        raise ValueError(f"Norm calibration {path} is missing valid p99_norm values for layers {missing}")
    return values

fixed_norm_values = _load_p99_norms(NORM_FILE, FIXED_ROUTE)
candidate_norm_values = _load_p99_norms(CANDIDATE_NORMS, CANDIDATE_LAYERS)

for path in ROLLOUT_FILES:
    with path.open("r", encoding="utf-8") as stream:
        first = stream.readline()
    if not first.strip():
        raise ValueError(f"Rollout shard is empty: {path}")
    json.loads(first)

alpha_data = json.loads(ALPHA_REPORT.read_text(encoding="utf-8"))
if not isinstance(alpha_data.get("shots"), dict) or not alpha_data["shots"]:
    raise ValueError("Saved alpha calibration report is incomplete")

bank_state = torch.load(BANK_CHECKPOINT, map_location="cpu", weights_only=True)
bank_digest = sha256_file(BANK_CHECKPOINT)
if bank_state.get("format") != "ars.independent_route_expert_bank" or bank_state.get("format_version") != 1:
    raise ValueError("Saved route-bank checkpoint has an unsupported format")
if not bank_state.get("training_complete") or not bank_state.get("route_experts_independent"):
    raise ValueError("Saved route-bank checkpoint is incomplete or shares expert weights")
if list(bank_state.get("candidate_layers", [])) != CANDIDATE_LAYERS:
    raise ValueError("Saved route bank and handoff candidate layers differ")
bank_routes = normalize_route_catalog(bank_state.get("route_catalog", []), CANDIDATE_LAYERS)
validate_route_expert_states(bank_routes, bank_state.get("route_experts", {}))
route_catalog = bank_state.get("curriculum", {}).get("router_catalog", bank_state.get("curriculum", {}).get("catalog", []))
if not route_catalog:
    raise ValueError("Saved bank has no router route catalog")

static_action = json.loads(STATIC_ACTION.read_text(encoding="utf-8"))
if static_action.get("bank_sha256") != bank_digest:
    raise ValueError("Saved static-bank action was selected for a different bank checkpoint")
if tuple(static_action.get("route", [])) not in {tuple(route) for route in route_catalog}:
    raise ValueError("Saved static-bank route is absent from the saved router catalog")
if not 1.0 <= float(static_action.get("alpha", 0.0)) <= 2.0:
    raise ValueError("Saved static-bank action has an unsupported gain")

REFERENCE_TEACHER_FORWARDS = 121348
REFERENCE_TRAIN_QUESTIONS = 1345
OBSERVED_FORWARDS_PER_SECOND = 4.9
estimated_teacher_forwards = REFERENCE_TEACHER_FORWARDS * MAX_ROUTER_ROWS / REFERENCE_TRAIN_QUESTIONS
estimated_teacher_hours = estimated_teacher_forwards / OBSERVED_FORWARDS_PER_SECOND / 3600
print({
    "cuda_visible_gpus": GPU_COUNT,
    "reuse_bank": True,
    "reuse_static_action": True,
    "reuse_alpha_calibration": True,
    "reuse_rollout_shards": True,
    "route_catalog_routes": len(route_catalog),
    "teacher_action_count_estimate": 1 + (len(route_catalog) - 1) * len(ROUTER_ALPHAS),
    "fixed_norm_layers_validated": len(fixed_norm_values),
    "candidate_norm_layers_validated": len(candidate_norm_values),
    "max_router_question_ids": MAX_ROUTER_ROWS,
    "router_epochs": ROUTER_EPOCHS,
    "estimated_teacher_forwards": round(estimated_teacher_forwards),
    "estimated_teacher_scoring_hours": round(estimated_teacher_hours, 2),
    "estimated_total_hours_with_overhead": round(estimated_teacher_hours + 0.4, 2),
}, flush=True)


## Run only router teacher scoring and fitting

This stage loads the existing bank and uses the already saved static action directly. It does not call bank training, alpha calibration, or static-bank screening. The teacher scores at most 400 question IDs, then the router is trained for three epochs and saves its checkpoint/report/teacher targets in the new writable output folder.


In [ ]:
import contextlib, io, time
from IPython.display import display

import scripts.run_session3_router as router_runner

ROUTER_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "logs").mkdir(parents=True, exist_ok=True)
router_log = RUN_DIR / "logs/router_training_resume.log"

def _router_checkpoint_complete():
    report_path = ROUTER_REPORT
    target_path = TEACHER_TARGETS
    if not (ROUTER_CHECKPOINT.is_file() and report_path.is_file() and target_path.is_file()):
        return False
    try:
        state = torch.load(ROUTER_CHECKPOINT, map_location="cpu", weights_only=True)
        report = json.loads(report_path.read_text(encoding="utf-8"))
        return (
            state.get("format") == "ars_catalog_router_v2"
            and state.get("bank_sha256") == bank_digest
            and list(state.get("candidate_layers", [])) == CANDIDATE_LAYERS
            and bool(report)
        )
    except (OSError, ValueError, RuntimeError, TypeError):
        return False

if _router_checkpoint_complete():
    ROUTER_STATUS = "reused_complete_router_checkpoint"
    print("A complete router checkpoint already matches this bank; skipping router training.", flush=True)
else:
    existing_targets = TEACHER_TARGETS
    if existing_targets.exists():
        print("A teacher-target file exists without a complete router checkpoint. The pinned runner does not resume partial teacher scoring; it will rescore the capped question set.", flush=True)

    # Reuse the recorded validation choice instead of spending time selecting it again.
    router_runner.select_static_bank_action = lambda *args, **kwargs: dict(static_action)
    args = router_runner.parse_args([
        "--stage", "train_router",
        "--output_dir", str(RUN_DIR),
        "--candidate_layers", *map(str, CANDIDATE_LAYERS),
        "--norm_calibration", str(CANDIDATE_NORMS),
        "--fixed_adapter_norm_calibration", str(NORM_FILE),
        "--fixed_adapter_checkpoint", str(CHECKPOINT),
        "--bank_checkpoint", str(BANK_CHECKPOINT),
        "--router_checkpoint", str(ROUTER_CHECKPOINT),
        "--rollout_files", *map(str, ROLLOUT_FILES),
        "--epochs", str(ROUTER_EPOCHS),
        "--max_questions", str(MAX_ROUTER_QUESTIONS),
        "--max_rows_per_question", str(COLLECTION_POLICY["max_training_rows_per_question"]),
        "--max_router_rows", str(MAX_ROUTER_ROWS),
        "--route_cost", str(ROUTE_COST),
        "--router_alphas", *map(str, ROUTER_ALPHAS),
        "--static_screen_questions", "0",
        "--task_validation_rows", str(TASK_VALIDATION_ROWS),
        "--seed", str(SEED),
        "--diversify_prompts",
    ])

    class _Tee(io.TextIOBase):
        def __init__(self, *streams): self.streams = streams
        def write(self, value):
            for stream in self.streams: stream.write(value); stream.flush()
            return len(value)
        def flush(self):
            for stream in self.streams: stream.flush()

    started = time.time()
    with router_log.open("w", encoding="utf-8") as log_stream:
        with contextlib.redirect_stdout(_Tee(sys.stdout, log_stream)), contextlib.redirect_stderr(_Tee(sys.stderr, log_stream)):
            exit_code = router_runner.run(args)
    elapsed_hours = (time.time() - started) / 3600
    if exit_code != 0 or not _router_checkpoint_complete():
        raise RuntimeError(f"Router stage did not complete successfully (exit_code={exit_code}). See {router_log} and keep this output for recovery.")
    ROUTER_STATUS = "passed"
    print({"router_status": ROUTER_STATUS, "elapsed_hours": round(elapsed_hours, 2), "checkpoint": str(ROUTER_CHECKPOINT), "teacher_targets": str(TEACHER_TARGETS)}, flush=True)


## Validate and package old plus new artifacts

The next benchmark notebooks expect a root `manifest.json` with `format=ars_training_bundle_v1` and `training_ready=true`. This cell preserves the original fixed-adapter manifest under `provenance/`, records the router-only continuation settings and checksums, and creates one ZIP containing the previous 01b files plus the completed router output.


In [ ]:
import math
from src.steering.route_bank import normalize_route_catalog, validate_route_expert_states

source_manifest_copy = RUN_DIR / "provenance/source_fixed_adapter_manifest.json"
source_manifest_copy.parent.mkdir(parents=True, exist_ok=True)
if not source_manifest_copy.exists():
    shutil.copy2(RUN_DIR / "manifest.json", source_manifest_copy)

required_paths = {
    "checkpoints/fixed_route_adapter.pt": RUN_DIR / "checkpoints/fixed_route_adapter.pt",
    "checkpoints/session3_steering_bank.pt": BANK_CHECKPOINT,
    "checkpoints/session3_router.pt": ROUTER_CHECKPOINT,
    "calibration/fixed_route_p99_norms.json": NORM_FILE,
    "calibration/candidate_norms_p99.json": CANDIDATE_NORMS,
    "selection/session3_static_bank_action.json": STATIC_ACTION,
    "training/alpha_calibration.json": ALPHA_REPORT,
}
missing = [name for name, path in required_paths.items() if not path.is_file() or not path.stat().st_size]
if missing:
    raise FileNotFoundError(f"Cannot create a complete training bundle; missing: {missing}")

router_state = torch.load(ROUTER_CHECKPOINT, map_location="cpu", weights_only=True)
if router_state.get("format") != "ars_catalog_router_v2" or router_state.get("bank_sha256") != bank_digest:
    raise ValueError("Router checkpoint is incompatible with the saved route bank")
router_report_path = ROUTER_REPORT
teacher_targets_path = TEACHER_TARGETS
if not router_report_path.is_file() or not teacher_targets_path.is_file():
    raise FileNotFoundError("Router checkpoint is missing its training report or complete teacher-target cache")

bank_state = torch.load(BANK_CHECKPOINT, map_location="cpu", weights_only=True)
bank_routes = normalize_route_catalog(bank_state.get("route_catalog", []), CANDIDATE_LAYERS)
validate_route_expert_states(bank_routes, bank_state.get("route_experts", {}))
bank_summary = {
    "format": bank_state["format"],
    "format_version": bank_state["format_version"],
    "architecture_id": bank_state["architecture_id"],
    "architecture_version": bank_state["architecture_version"],
    "route_catalog": bank_state["route_catalog"],
    "router_catalog": route_catalog,
    "route_expert_count": len(bank_state["route_experts"]),
    "route_experts_independent": True,
    "route_expert_parameter_count": bank_state["route_expert_parameter_count"],
}

PHASE_STATUS = dict(SOURCE_MANIFEST.get("phase_status", {}))
PHASE_STATUS.update({
    "runtime": "passed",
    "norms": "passed",
    "collection": "completed",
    "training": "passed",
    "candidate_norms": "passed",
    "alpha_calibration": "passed",
    "steering_bank": "passed",
    "router": "passed",
    "packaging": "pending",
})
required_phase_names = ("runtime", "tests", "norms", "collection", "training", "candidate_norms", "alpha_calibration", "steering_bank", "router")
accepted_status = {
    "runtime": {"passed", "warning_single_gpu"},
    "collection": {"completed", "passed"},
    **{name: {"passed"} for name in required_phase_names if name not in {"runtime", "collection"}},
}
failed_phases = {
    key: PHASE_STATUS.get(key, "not_run") for key in required_phase_names
    if PHASE_STATUS.get(key) not in accepted_status[key]
}
missing_code = [
    name for name, metadata in SOURCE_MANIFEST.get("code", {}).items()
    if not (RUN_DIR / name).is_file() or sha256_file(RUN_DIR / name) != metadata["sha256"]
]
training_ready = not missing and not missing_code and not failed_phases
PHASE_STATUS["packaging"] = "passed" if training_ready else f"partial: missing={missing + missing_code}, phases={failed_phases}"

for archive_name, source_path in required_paths.items():
    destination = RUN_DIR / archive_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        shutil.copy2(source_path, destination)

code_snapshot = {name: {"sha256": metadata["sha256"]} for name, metadata in SOURCE_MANIFEST.get("code", {}).items()}
artifact_hashes = {name: {"sha256": sha256_file(RUN_DIR / name)} for name in required_paths}
artifact_hashes["checkpoints/session3_router_training.json"] = {"sha256": sha256_file(router_report_path)}
artifact_hashes["checkpoints/session3_route_teacher_targets.pt"] = {"sha256": sha256_file(teacher_targets_path)}

manifest = {
    "format": "ars_training_bundle_v1",
    "training_ready": training_ready,
    "architecture_id": SOURCE_MANIFEST["architecture_id"],
    "architecture_version": SOURCE_MANIFEST["architecture_version"],
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": RUN_ID,
    "source_run_id": SOURCE_RUN_ID,
    "fixed_adapter_source_run_id": SOURCE_MANIFEST.get("run_id"),
    "code_commit": SOURCE_MANIFEST["code_commit"],
    "parent_fixed_adapter_manifest_sha256": sha256_file(source_manifest_copy),
    "model": SOURCE_MANIFEST["model"],
    "route": FIXED_ROUTE,
    "candidate_layers": CANDIDATE_LAYERS,
    "rank": SOURCE_MANIFEST["rank"],
    "rho_max": SOURCE_MANIFEST["rho_max"],
    "norm_stat": SOURCE_MANIFEST["norm_stat"],
    "seed": SEED,
    "shot_weights": SOURCE_MANIFEST["shot_weights"],
    "training_profile": {
        "name": "router_only_resume",
        "bank_reused": True,
        "static_action_reused": True,
        "alpha_calibration_reused": True,
        "router_teacher_question_cap": MAX_ROUTER_ROWS,
        "router_epochs": ROUTER_EPOCHS,
        "router_alphas": ROUTER_ALPHAS,
        "route_cost": ROUTE_COST,
        "task_validation_rows": TASK_VALIDATION_ROWS,
    },
    "steering_bank": bank_summary,
    "phase_status": PHASE_STATUS,
    "required_phase_status": {key: PHASE_STATUS.get(key) for key in required_phase_names},
    "missing_required": missing + missing_code,
    "failed_phases": failed_phases,
    "artifacts": artifact_hashes,
    "code": code_snapshot,
}
manifest_path = RUN_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
(RUN_DIR / "phase_status.json").write_text(json.dumps(PHASE_STATUS, indent=2) + "\n", encoding="utf-8")

final_bundle = RUN_DIR / "ars_training_bundle.zip"
with zipfile.ZipFile(final_bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob("*")):
        if not path.is_file() or path.resolve() == final_bundle.resolve() or path.name == "ars_training_bundle.zip":
            continue
        archive.write(path, path.relative_to(RUN_DIR).as_posix())

print(json.dumps({
    "training_ready": training_ready,
    "phase_status": PHASE_STATUS,
    "failed_phases": failed_phases,
    "missing": missing + missing_code,
    "router_checkpoint": str(ROUTER_CHECKPOINT),
    "route_teacher_targets": str(teacher_targets_path),
    "complete_bundle": str(final_bundle),
    "bundle_size_mb": round(final_bundle.stat().st_size / (1024 * 1024), 2),
    "included_previous_output": True,
}, indent=2), flush=True)
if not training_ready:
    raise RuntimeError("The combined ZIP was created for diagnosis but is not benchmark-ready; inspect the status above.")
try:
    from IPython.display import FileLink, display
    display(FileLink(str(final_bundle)))
except Exception:
    pass


## Expected result

When successful, download `ars_training_bundle.zip` from the output directory and attach it to the evaluation notebooks. It contains the prior 01b handoff artifacts, rollout shards and logs, plus the newly completed router checkpoint, teacher targets, router report, updated manifest, and phase status.

The saved bank and alpha artifacts are copied byte-for-byte from the old run. The continuation changes only the router training sample cap and its newly generated router artifacts.
